### Summer Working Connections 2026
### Fundamentals of Quantum Programming
### Final Assessment submitted by Hugh N.

- Follow the steps provided below to complete your final assessment to earn the workshop Credly badge.
- **You may use any code examples provided in the workshop notebooks as necessary to implement your solution.**
- Use the code block provided to enter and execute your code. The setup block is provided before the template block to support import operations in your code.
- Once completed, you can either save your notebook to your forked repo in GitHub and provide me with the URL, or download the notebook (use the Colab menu bar: File/Download/Download .ipynb) and email the download notebook to me so I can view and execute your code.


### Rubric
- Your work will be scored based on completion of the following tasks.
- **Partial credit will be given! (don't leave anything blank)**
- A minimum score of **80%** is required to earn the badge.
<br>

- Initialization + State Table (20%): correct 3-qubit superposition and pre-oracle state table
- Phase Oracle (20%): correct two-target oracle and post-oracle state table
- Grover Operator (20%): one correct Grover iteration with visible amplitude change
- Measurement (20%): runs with shots = 1000; marked states appear more frequently
- Complete, Working Program (20%): runs without errors; all required outputs shown

## Final Assessment: Grover Search with Two Targets
### Objective
- Use a quantum circuit to identify target values from a set of possibilities by:
  - encoding the problem as a phase oracle
  - applying the Grover operator
  - analyzing state tables and measurement results
- Problem Description
  - You are given a “database” of all 3-qubit outcomes (000, 001, 010, 011, 100, 101, 110, 111)
  - Your task is to mark two target values and observe how the circuit behavior changes.
  - Use k = 3 and k = 6 as the marked outcomes and start with a uniform superposition
- Steps
  0. Run the setup code block
  1. Define the target predicate for marked outcomes (k = 3, 6)
  2. Define the phase oracle to flip amplitudes of marked states
  3. Define the diffusion operator (inversion about the mean)
  4. Initialize a 3-qubit circuit and apply Hadamard gates to create a uniform superposition
  5. Run the circuit and display the initial state table
  6. Copy the original state to preserve it for the Grover iteration
  7. Apply the oracle to the current state and display the result
  8. Apply one Grover iteration (oracle + diffusion) using the saved original state
  9. Measure the final state over multiple shots and display counts
###Implementation Notes
- Use the provided object-oriented framework and classical functions for the oracle and grover implementation
  - QuantumRegister
  - QuantumCircuit
  - oracle(...)
  - run()
- Use:
  - print_state_table(...) for output
  - the provided measure(...) function for sampling
### Deliverables
1. Code
  - Submit a complete working program in the provided code block in the assessment notebook (refresh your repo fork, then access the notebook in the refreshed content).
2. Output
  - State table before oracle
  - State table after oracle
  - State table after one Grover iteration
  - Measurement results (shots = 1000)



In [23]:
# ---- setup for clone/repo imports ----
import os
import sys
import subprocess
import importlib

REPO_URL = "https://github.com/learnqc/code.git"
REPO_DIR = "/content/code"
SRC_DIR = f"{REPO_DIR}/src"

# Install required pip package(s)
subprocess.run(
    ["pip", "install", "-q", "sty"],
    check=True
)

# Clone repo if needed
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear any stale imports
importlib.invalidate_caches()

print("Setup complete")

Setup complete


In [24]:
# My solution starts here

In [25]:
# need these shown as shown Day 4 ipynb
from math import sqrt
from ch03.util import *          # print_state_table
from ch05.sim_circuit import *   # QuantumRegister, QuantumCircuit, measure

# step 1: Define the target predicate for marked outcomes (k = 3, 6) ; marking the outcomes, 011 and 110


items = [3, 6]

def is_target(k):
    return k in items

predicate = is_target

print(f"Marked outcomes: {items}")

Marked outcomes: [3, 6]


In [26]:
# step 2: Define the phase oracle to flip amplitudes of marked states; classical phase oracle: multiplies amplitude of each good outcome by -1
def oracle(state, predicate):
    for k in range(len(state)):
        if predicate(k):
            state[k] *= -1

In [27]:
# step 3; Define the diffusion operator (inversion about the mean)
# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# inversion operator: current state around the original state
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# one full Grover iterate G = oracle followed by diffusion
def grover_iteration(state, original, predicate):
    oracle(state, predicate)
    inversion(original, state)

In [28]:
# step 4: Initialize a 3-qubit circuit and apply Hadamard gates to create a uniform superposition
n = 3
q = QuantumRegister(n)
qc = QuantumCircuit(q)

for i in range(n):
    qc.h(q[i])

In [29]:
# step 5: Run the circuit and display the initial (pre-oracle) state table
initial_state = qc.run()

print("STATE TABLE - Before Oracle (uniform superposition)")
print_state_table(initial_state)

STATE TABLE - Before Oracle (uniform superposition)

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
1        001     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
2        010     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
3        011     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
4        100     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
5        101     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
6        110     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
7        111     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 



In [30]:
# step 6: Copy the original state to preserve it for the Grover iteration
original_state = initial_state.copy()
state = initial_state.copy()

In [31]:
#step 7: Apply the oracle to the current state and display the result
oracle(state, predicate)

print("STATE TABLE - After Oracle (targets 3 & 6 phase-flipped)")
print_state_table(state)

STATE TABLE - After Oracle (targets 3 & 6 phase-flipped)

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
1        001     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
2        010     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
3        011    -0.3536 + i0.0000     180.00°   0.3536     ████████                  0.125 
4        100     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
5        101     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
6        110    -0.3536 + i0.0000     180.00°   0.3536     ████████                  0.125 
7        111     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 



In [32]:
# step 8: Apply one Grover iteration (oracle + diffusion) using the saved original state
state = original_state.copy()
grover_iteration(state, original_state, predicate)

print("STATE TABLE - After 1 Grover Iteration (Oracle + Diffusion)")
print_state_table(state)

# sanity checks
total_prob = sum(abs(a) ** 2 for a in state)
assert abs(total_prob - 1) < 1e-9, "State is not normalized!"

good_prob = sum(abs(state[k]) ** 2 for k in items)
print(f"\nCombined probability of marked states {items} after 1 iteration: "
      f"{good_prob:.4f}  (baseline was {len(items) / 2**n:.4f})")
assert good_prob > len(items) / 2**n, "The Grover iteration didn't amplify the marked states"

STATE TABLE - After 1 Grover Iteration (Oracle + Diffusion)

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.0000 + i0.0000               0.0                                  0.0   
1        001     0.0000 + i0.0000               0.0                                  0.0   
2        010     0.0000 + i0.0000               0.0                                  0.0   
3        011     0.7071 + i0.0000       0.00°   0.7071     ████████████████          0.5   
4        100     0.0000 + i0.0000               0.0                                  0.0   
5        101     0.0000 + i0.0000               0.0                                  0.0   
6        110     0.7071 + i0.0000       0.00°   0.7071     ████████████████          0.5   
7        111     0.0000 + i0.0000               0.0                                  0.0   


Combine

In [33]:
# step 9: Measure the final state over multiple shots and display counts
shots = 1000
samples = measure(state, shots)

print(f"Final measurement results ({shots} shots)")
for outcome in sorted(samples.keys()):
    bits = format(outcome, f"0{n}b")
    marker = "  <-- marked" if outcome in items else ""
    print(f"  Outcome {outcome} ({bits}): {samples[outcome]:4d} counts{marker}")

marked_counts = sum(samples.get(k, 0) for k in items)
unmarked_avg = (shots - marked_counts) / (2**n - len(items))
marked_avg = marked_counts / len(items)

print(f"\nTotal counts on marked states {items}: {marked_counts} / {shots} "
      f"({100*marked_counts/shots:.1f}%)")
print(f"Average counts per marked outcome:   {marked_avg:.1f}")
print(f"Average counts per unmarked outcome: {unmarked_avg:.1f}")
assert marked_avg > unmarked_avg, "Marked states were not measured more frequently"

print("\nAll checks passed: marked states 3 and 6 appeared more often.")

Final measurement results (1000 shots)
  Outcome 3 (011):  514 counts  <-- marked
  Outcome 6 (110):  486 counts  <-- marked

Total counts on marked states [3, 6]: 1000 / 1000 (100.0%)
Average counts per marked outcome:   500.0
Average counts per unmarked outcome: 0.0

All checks passed: marked states 3 and 6 appeared more often.
